## Week 6 exercise — Study coach MCP

This exercise adds a small **custom MCP server** (`study_server.py`) that lists and reads **`*.md` files** in this folder (`6_mcp/IbrahimSheriff/`). The **OpenAI Agents SDK** `Agent` uses `MCPServerStdio` to call those tools—same pattern as Week 6 Lab 2 (`accounts_server.py`)—with the chat model served through **OpenRouter** (OpenAI-compatible API).

**Goals:**

1. Implement tools: list note filenames and read one file (paths constrained to this folder).
2. Connect the server with `uv run study_server.py` and list tools from the notebook.
3. Run an agent that plans which note to open and answers a question about your study material.

### Setup

Markdown notes (`mcp_basics.md`, `agents_sdk.md`, …) sit **next to** `study_server.py` in this folder. The notebook sets **`cwd`** on `MCPServerStdio` so `uv run study_server.py` runs from `6_mcp/IbrahimSheriff` even if the kernel’s current directory is the repo root.

Ensure `OPENROUTER_API_KEY` is in your `.env` at the repo root. The Agents SDK uses an OpenAI-compatible client pointed at [OpenRouter](https://openrouter.ai).

In [ ]:
import os

from dotenv import load_dotenv
from agents import Agent, AsyncOpenAI, Runner, trace
from agents.mcp import MCPServerStdio
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel
from IPython.display import Markdown, display

load_dotenv(override=True)

In [ ]:
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_MODEL = "openai/gpt-4o-mini"

openrouter_client = AsyncOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=OPENROUTER_BASE_URL,
    default_headers={
        "HTTP-Referer": "http://localhost",
        "X-Title": "Week6-StudyCoach",
    },
)

model = OpenAIChatCompletionsModel(
    model=OPENROUTER_MODEL,
    openai_client=openrouter_client,
)

### 1. Inspect tools exposed by `study_server.py`

The server registers `list_study_notes` and `read_study_note` (see `IbrahimSheriff/study_server.py`).

In [ ]:
from pathlib import Path


def _exercise_dir() -> Path:
    """Folder that contains study_server.py and the *.md notes (works from repo root or this folder)."""
    cwd = Path.cwd().resolve()
    if (cwd / "study_server.py").exists():
        return cwd
    for base in [cwd, *cwd.parents]:
        cand = base / "6_mcp" / "IbrahimSheriff"
        if (cand / "study_server.py").exists():
            return cand
    raise FileNotFoundError(
        "Could not find 6_mcp/IbrahimSheriff/study_server.py — cd to IbrahimSheriff or open this notebook from that folder."
    )


EXERCISE_DIR = _exercise_dir()
params = {
    "command": "uv",
    "args": ["run", "study_server.py"],
    "cwd": str(EXERCISE_DIR),
}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    study_tools = await server.list_tools()

study_tools

### 2. Agent that uses your notes

Ask for a short study plan grounded in the `*.md` files in this folder.

In [ ]:
instructions = (
    "You help the user study. Use the study tools: first list available notes, "
    "then read the note(s) needed to answer. Be concise and cite which file you used."
)
request = (
    "What are the main ideas in my notes about MCP and the Agents SDK, "
    "and what should I review first?"
)

In [ ]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(
        name="study_coach",
        instructions=instructions,
        model=model,
        mcp_servers=[mcp_server],
    )
    with trace("study_coach"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

### Stretch ideas

- Add a third tool (e.g. word count or a note tag) in `study_server.py`.
- Combine this server with another `MCPServerStdio` (memory or fetch from Lab 1 / 3) and pass both in `mcp_servers=[...]`.
- Add another `.md` file next to `study_server.py` and re-run the agent.